In [2]:
import pandas as pd
import numpy as np
import os

# Menampilkan semua kolom pandas tanpa terpotong
pd.set_option('display.max_columns', None)

In [3]:
# Path folder raw data
raw_path = '../data/raw/'

# Load semua dataset ke dalam dictionary DataFrame
datasets = {
    'customers': pd.read_csv(f'{raw_path}olist_customers_dataset.csv'),
    'geolocation': pd.read_csv(f'{raw_path}olist_geolocation_dataset.csv'),
    'order_items': pd.read_csv(f'{raw_path}olist_order_items_dataset.csv'),
    'order_payments': pd.read_csv(f'{raw_path}olist_order_payments_dataset.csv'),
    'order_reviews': pd.read_csv(f'{raw_path}olist_order_reviews_dataset.csv'),
    'orders': pd.read_csv(f'{raw_path}olist_orders_dataset.csv'),
    'products': pd.read_csv(f'{raw_path}olist_products_dataset.csv'),
    'sellers': pd.read_csv(f'{raw_path}olist_sellers_dataset.csv'),
    'category_translation': pd.read_csv(f'{raw_path}product_category_name_translation.csv')
}

print("Semua dataset berhasil dimuat!")

Semua dataset berhasil dimuat!


In [4]:
# Memeriksa baris, kolom, missing values, dan duplikat dari tiap tabel
for name, df in datasets.items():
    print(f"=== DATASET: {name.upper()} ===")
    print(f"Jumlah Baris & Kolom: {df.shape}")
    print(f"Jumlah Duplikat     : {df.duplicated().sum()}")
    print("Missing Values      :")
    nulls = df.isnull().sum()
    print(nulls[nulls > 0] if nulls.sum() > 0 else "Tidak ada missing value")
    print("-" * 50)

=== DATASET: CUSTOMERS ===
Jumlah Baris & Kolom: (99441, 5)
Jumlah Duplikat     : 0
Missing Values      :
Tidak ada missing value
--------------------------------------------------
=== DATASET: GEOLOCATION ===
Jumlah Baris & Kolom: (1000163, 5)
Jumlah Duplikat     : 261831
Missing Values      :
Tidak ada missing value
--------------------------------------------------
=== DATASET: ORDER_ITEMS ===
Jumlah Baris & Kolom: (112650, 7)
Jumlah Duplikat     : 0
Missing Values      :
Tidak ada missing value
--------------------------------------------------
=== DATASET: ORDER_PAYMENTS ===
Jumlah Baris & Kolom: (103886, 5)
Jumlah Duplikat     : 0
Missing Values      :
Tidak ada missing value
--------------------------------------------------
=== DATASET: ORDER_REVIEWS ===
Jumlah Baris & Kolom: (99224, 7)
Jumlah Duplikat     : 0
Missing Values      :
review_comment_title      87656
review_comment_message    58247
dtype: int64
--------------------------------------------------
=== DATASET: ORDERS 

In [5]:
# 1. Konversi Kolom Tanggal di Tabel ORDERS
date_cols_orders = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]
for col in date_cols_orders:
    datasets['orders'][col] = pd.to_datetime(datasets['orders'][col])

# 2. Konversi Kolom Tanggal di Tabel REVIEWS & ITEMS
datasets['order_reviews']['review_creation_date'] = pd.to_datetime(datasets['order_reviews']['review_creation_date'])
datasets['order_reviews']['review_answer_timestamp'] = pd.to_datetime(datasets['order_reviews']['review_answer_timestamp'])
datasets['order_items']['shipping_limit_date'] = pd.to_datetime(datasets['order_items']['shipping_limit_date'])

# 3. Menangani Missing Value pada Kategori Produk (Translate ke Bahasa Inggris)
# Merge tabel products dengan translation
products_clean = datasets['products'].merge(
    datasets['category_translation'], 
    on='product_category_name', 
    how='left'
)
# Isi kategori yang hilang dengan 'unknown'
products_clean['product_category_name_english'] = products_clean['product_category_name_english'].fillna('unknown')
datasets['products'] = products_clean

# 4. Hapus Duplikat pada Geolocation (Ambil 1 ZIP Code = 1 Lat/Lng unik)
datasets['geolocation'] = datasets['geolocation'].drop_duplicates(subset=['geolocation_zip_code_prefix'])

print("Proses Data Cleaning selesai!")

Proses Data Cleaning selesai!


In [6]:
cleaned_path = '../data/cleaned/'
os.makedirs(cleaned_path, exist_ok=True)

# Simpan semua dataframe yang sudah dibersihkan
for name, df in datasets.items():
    output_file = f'{cleaned_path}cleaned_{name}.csv'
    df.to_csv(output_file, index=False)
    print(f"Tersimpan: {output_file}")

Tersimpan: ../data/cleaned/cleaned_customers.csv
Tersimpan: ../data/cleaned/cleaned_geolocation.csv
Tersimpan: ../data/cleaned/cleaned_order_items.csv
Tersimpan: ../data/cleaned/cleaned_order_payments.csv
Tersimpan: ../data/cleaned/cleaned_order_reviews.csv
Tersimpan: ../data/cleaned/cleaned_orders.csv
Tersimpan: ../data/cleaned/cleaned_products.csv
Tersimpan: ../data/cleaned/cleaned_sellers.csv
Tersimpan: ../data/cleaned/cleaned_category_translation.csv
